In [13]:
import unicodedata
import re
import pandas as pd
import numpy as np
from difflib import SequenceMatcher

In [7]:
MBIC_raw = pd.read_excel("~/BABE/Neural-Media-Bias-Detection-Using-Distant-Supervision-With-BABE/data/raw_labels_MBIC.xlsx")
SG1_raw = pd.read_excel("~/BABE/Neural-Media-Bias-Detection-Using-Distant-Supervision-With-BABE/data/raw_labels_SG1.xlsx")
SG2_raw = pd.read_excel("~/BABE/Neural-Media-Bias-Detection-Using-Distant-Supervision-With-BABE/data/raw_labels_SG2.xlsx")

MBIC_final = pd.read_excel("~/BABE/Neural-Media-Bias-Detection-Using-Distant-Supervision-With-BABE/data/final_labels_MBIC.xlsx")
SG1_final = pd.read_excel("~/BABE/Neural-Media-Bias-Detection-Using-Distant-Supervision-With-BABE/data/final_labels_SG1.xlsx")
SG2_final = pd.read_excel("~/BABE/Neural-Media-Bias-Detection-Using-Distant-Supervision-With-BABE/data/final_labels_SG2.xlsx")

/ihome/xli/sek188/MSTHESIS/envs/.venv/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [3]:
SG1_raw.columns

Index(['text', 'news_link', 'outlet', 'topic', 'type', 'label_bias',
       'label_opinion', 'biased_words', 'Label_bias_0-1', 'annotator_id'],
      dtype='str')

In [4]:
SG2_raw.columns

Index(['text', 'news_link', 'outlet', 'topic', 'type', 'label_bias',
       'label_opinion', 'biased_words', 'annotator_id', 'Label_bias_0-1',
       'df_id'],
      dtype='str')

In [9]:
MBIC_raw.columns

Index(['survey_record_id', 'sentence_id', 'sentence_group_id', 'created_at',
       'label_bias', 'words', 'label_opinion', 'group_id', 'text', 'news_link',
       'type', 'topic', 'outlet', 'mturk_id', 'age', 'gender', 'education',
       'native_english_speaker', 'political_ideology', 'followed_news_outlets',
       'news_check_frequency', 'survey_completed'],
      dtype='str')

In [10]:
SG2_final.columns

Index(['text', 'news_link', 'outlet', 'topic', 'type', 'label_bias',
       'label_opinion', 'biased_words'],
      dtype='str')

In [14]:
def normalize_text(x):
    """Normalize text for matching, not for display."""
    if pd.isna(x):
        return pd.NA
    
    x = str(x)
    x = unicodedata.normalize("NFKC", x)
    x = x.replace("“", '"').replace("”", '"').replace("’", "'")
    x = re.sub(r"\s+", " ", x).strip().lower()
    return x if x else pd.NA

In [15]:
def prep_raw(df, dataset_name):
    df = df.copy()
    df["dataset"] = dataset_name
    df["_source_row"] = np.arange(len(df))
    df["_norm_text"] = df["text"].map(normalize_text)

    # useful stable source id, separate from joined_sent_id
    if "sentence_id" in df.columns:
        df["source_sentence_id"] = df["sentence_id"]
    elif "df_id" in df.columns:
        df["source_sentence_id"] = dataset_name.upper() + "_" + df["df_id"].astype(str)
    else:
        df["source_sentence_id"] = dataset_name.upper() + "_" + (df["_source_row"] + 1).astype(str)

    return df


raw_all = pd.concat(
    [
        prep_raw(MBIC_raw, "mbic"),
        prep_raw(SG1_raw, "sg1"),
        prep_raw(SG2_raw, "sg2"),
    ],
    ignore_index=True,
    sort=False
)

In [16]:
def first_non_null_or_join(s):
    vals = (
        s.dropna()
         .astype(str)
         .map(str.strip)
    )
    vals = vals[vals.ne("")]
    vals = vals.unique()

    if len(vals) == 0:
        return pd.NA
    if len(vals) == 1:
        return vals[0]
    return " | ".join(vals)   # flags conflicts without destroying info


def prep_final(df, dataset_name):
    df = df.copy()
    df["dataset"] = dataset_name
    df["_norm_text"] = df["text"].map(normalize_text)

    out = df[["dataset", "_norm_text", "label_bias"]].copy()
    out = out.rename(columns={"label_bias": "consensus_label"})

    return out


final_all = pd.concat(
    [
        prep_final(MBIC_final, "mbic"),
        prep_final(SG1_final, "sg1"),
        prep_final(SG2_final, "sg2"),
    ],
    ignore_index=True,
    sort=False
)

final_lookup = (
    final_all
    .dropna(subset=["_norm_text"])
    .groupby(["dataset", "_norm_text"], as_index=False)
    .agg(consensus_label=("consensus_label", first_non_null_or_join))
)

In [17]:
df_all = raw_all.merge(
    final_lookup,
    on=["dataset", "_norm_text"],
    how="left",
    validate="many_to_one"
)

In [18]:
def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()


def fuzzy_consensus_fill(df, final_lookup, threshold=0.995):
    df = df.copy()

    for dataset_name in df["dataset"].dropna().unique():
        final_sub = final_lookup[final_lookup["dataset"].eq(dataset_name)].copy()
        final_texts = final_sub["_norm_text"].dropna().unique().tolist()

        missing_mask = (
            df["dataset"].eq(dataset_name)
            & df["consensus_label"].isna()
            & df["_norm_text"].notna()
        )

        missing_texts = df.loc[missing_mask, "_norm_text"].dropna().unique()

        fuzzy_map = {}

        for txt in missing_texts:
            best_txt = None
            best_score = 0

            for candidate in final_texts:
                # skip obviously different lengths
                if abs(len(txt) - len(candidate)) / max(len(txt), len(candidate), 1) > 0.02:
                    continue

                score = similarity(txt, candidate)

                if score > best_score:
                    best_score = score
                    best_txt = candidate

            if best_txt is not None and best_score >= threshold:
                consensus = final_sub.loc[
                    final_sub["_norm_text"].eq(best_txt),
                    "consensus_label"
                ].iloc[0]

                fuzzy_map[txt] = consensus

        fill_mask = missing_mask & df["_norm_text"].isin(fuzzy_map)
        df.loc[fill_mask, "consensus_label"] = df.loc[fill_mask, "_norm_text"].map(fuzzy_map)

    return df


df_all = fuzzy_consensus_fill(df_all, final_lookup, threshold=0.995)

In [19]:
def build_joined_sent_id_map(texts, threshold=0.995):
    """
    Greedy canonicalization.
    Exact normalized matches are already collapsed by unique().
    Fuzzy matches only happen at very high similarity.
    """
    unique_texts = pd.Series(texts).dropna().drop_duplicates().tolist()

    canonical_reps = []
    text_to_rep = {}

    for txt in unique_texts:
        matched_rep = None
        best_score = 0

        for rep in canonical_reps:
            # avoid comparing wildly different lengths
            if abs(len(txt) - len(rep)) / max(len(txt), len(rep), 1) > 0.02:
                continue

            score = similarity(txt, rep)

            if score > best_score:
                best_score = score
                matched_rep = rep

        if matched_rep is not None and best_score >= threshold:
            text_to_rep[txt] = matched_rep
        else:
            canonical_reps.append(txt)
            text_to_rep[txt] = txt

    rep_to_id = {
        rep: f"JS_{i:06d}"
        for i, rep in enumerate(canonical_reps, start=1)
    }

    return {
        txt: rep_to_id[rep]
        for txt, rep in text_to_rep.items()
    }


joined_map = build_joined_sent_id_map(df_all["_norm_text"], threshold=0.995)

df_all["joined_sent_id"] = df_all["_norm_text"].map(joined_map)

In [20]:
front_cols = [
    "joined_sent_id",
    "dataset",
    "source_sentence_id",
    "consensus_label",
    "text",
]

other_cols = [c for c in df_all.columns if c not in front_cols]

df_all = df_all[front_cols + other_cols]

In [22]:
df_all.columns

Index(['joined_sent_id', 'dataset', 'source_sentence_id', 'consensus_label',
       'text', 'survey_record_id', 'sentence_id', 'sentence_group_id',
       'created_at', 'label_bias', 'words', 'label_opinion', 'group_id',
       'news_link', 'type', 'topic', 'outlet', 'mturk_id', 'age', 'gender',
       'education', 'native_english_speaker', 'political_ideology',
       'followed_news_outlets', 'news_check_frequency', 'survey_completed',
       '_source_row', '_norm_text', 'biased_words', 'Label_bias_0-1',
       'annotator_id', 'df_id'],
      dtype='str')

In [36]:
adf = df_all.copy()
adf['text'] = adf['text'].astype(str).fillna("")
adf["followed_news_outlets_clean"] = (
    adf["followed_news_outlets"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("'", "", regex=False)
    .str.replace('"', "", regex=False)
)
adf["num_foll_news_outlet"] = adf["followed_news_outlets_clean"].apply(
    lambda x: len([item for item in x.split(",") if item.strip() != ""])
)

adf["num_age"] = pd.cut(adf["age"],bins=[-np.inf, 18, 30, 45, 55, 65, np.inf],labels=False)

adf["num_poli_idealogy"] = pd.cut(adf["political_ideology"],
                                 bins=[-np.inf, -6, -1, 1, 5, np.inf],
                                 labels=False)

adf["num_foll_news_outlet"] = pd.cut(adf["num_foll_news_outlet"],
                                          bins=[-np.inf, 1, 2, 3, 4, 6, np.inf],
                                          labels=False)


adf["num_news_check_frequency"] = adf["news_check_frequency"].map({
    "Never": 0,
    "Very rarely": 1,
    "Several times per month": 2,
    "Several times per week": 3,
    "Every day": 4,
    "Several times per day": 5,
})

def bin_education(edu):
    edu = str(edu).strip()
    if edu in {"Graduate work"}:
        return 5
    if edu in {"BA", "Bachelor", "Bachelors", "Bachelor's degree", "Bachelor’s degree"}:
        return 4
    if edu in {"Associate degree"}:
        return 3
    if edu in {"Some college", "Vocational or technical school"}:
        return 2
    if edu == "High school graduate":
        return 1
    return 0

adf["num_education"] = adf["education"].apply(bin_education)
adf['num_gender'] = adf['gender'].map({
    'Male': 1, 'male': 1,
    'Female': 0, 'female': 0,
}).fillna(2).astype(int)
adf["label_binary"] = adf["label_bias"].str.startswith("Biased").astype(int)


def binary_entropy(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -p * np.log(p) - (1 - p) * np.log(1 - p)

def h_cond_given_demo(group, demo_col, label_col='label_binary'):
    """H(Y | D=demo_col) for one sentence's ratings."""
    total = len(group)
    if total == 0:
        return 0.0
    h = 0.0
    for _, sub in group.groupby(demo_col):
        n = len(sub)
        weight = n / total
        p_biased = sub[label_col].mean()
        h += weight * binary_entropy(p_biased)
    return h

out_cat_names = ["num_age", "num_gender", "num_education", 
                 "num_poli_idealogy", "num_foll_news_outlet"]

rows = []
for sent_id, group in adf.groupby('joined_sent_id'):  # whatever your sentence-id column is called
    n_raters = len(group)
    p_biased = group['label_binary'].mean()
    h_obs = binary_entropy(p_biased)
    dataset = group['dataset'].iloc[0]
    
    h_cond_by_demo = {demo: h_cond_given_demo(group, demo) for demo in out_cat_names}
    h_cond_avg = np.mean(list(h_cond_by_demo.values()))
    
    rows.append({
        'sentence_id': sent_id,
        'n_raters': n_raters,
        'p_biased': p_biased,
        'h_obs': h_obs,
        **{f'h_cond_{demo}': v for demo, v in h_cond_by_demo.items()},
        'h_cond_avg': h_cond_avg,
        'I_avg': h_obs - h_cond_avg,  # mutual information, averaged across demographics
        'dataset': dataset
    })

sentence_df = pd.DataFrame(rows)

In [37]:
sdf = sentence_df.copy()

In [38]:
sdf.shape

(3678, 12)

In [39]:
sentence_df['I_avg'].describe()

count    3.678000e+03
mean     1.620192e-01
std      1.089442e-01
min      5.955125e-12
25%      1.145240e-11
50%      2.001610e-01
75%      2.455638e-01
max      6.532905e-01
Name: I_avg, dtype: float64

In [50]:
sdf[sdf['dataset'] == 'mbic']['I_avg'].describe()


count    1.700000e+03
mean     2.061173e-01
std      7.937137e-02
min      5.955125e-12
25%      1.624057e-01
50%      2.131685e-01
75%      2.509660e-01
max      6.532905e-01
Name: I_avg, dtype: float64

In [51]:
sdf[sdf['dataset'] == 'sg2']['I_avg'].describe()

count    1.975000e+03
mean     1.243075e-01
std      1.163983e-01
min      1.145216e-11
25%      1.145240e-11
50%      2.001610e-01
75%      2.001610e-01
max      2.772589e-01
Name: I_avg, dtype: float64

In [46]:
sdf['dataset'].value_counts()

dataset
sg2     1975
mbic    1700
sg1        3
Name: count, dtype: int64